In [12]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import gc
import random
import warnings
import ctypes

In [13]:
def set_seed(seed: int) -> int:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    
    warnings.filterwarnings("ignore")
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)

    def seed_worker(worker_id: int) -> None:
        worker_seed = torch.initial_seed() % (2 ** 32)
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    
    print(f"Random seed: {seed}")
    return seed_worker

def clear_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except:
        pass

In [14]:
DATASET_CONFIG = {
    "sunspots": {
        "path": "/kaggle/input/datasets/hoaianhnguyenduc/sunspots/sunspots_dataset.csv",
        "time_col": "Date",
        "target_col": "Monthly Mean Total Sunspot Number",
        "type": "univariate"
    },
    "appliances_energy": {
        "path": "/kaggle/input/datasets/hoaianhnguyenduc/appliances/appliances_energy_dataset.csv",
        "time_col": "date",
        "target_col": "Appliances",  
        "type": "multivariate"
    },
    "beijing_air_quality": {
        "path": "/kaggle/input/datasets/hoaianhnguyenduc/beijing/beijing_air_quality_dataset.csv",
        "time_col": ["year", "month", "day", "hour"], 
        "target_col": "PM2.5",  
        "type": "multivariate"
    },
    "hanoi_air_quality": {
        "path": "/kaggle/input/datasets/hoaianhnguyenduc/hanoiair/hanoi_air_quality_dataset.csv",
        "time_col": "Local Time",
        "target_col": "AQI",  
        "type": "multivariate"
    },
    "bitcoin": {
        "path": "/kaggle/input/datasets/hoaianhnguyenduc/bitcoin/bitcoin_dataset.csv",
        "time_col": "Timestamp",
        "target_col": "Open",  
        "type": "univariate"
    }   
}

In [15]:
class TSWindowDataset(Dataset):
    def __init__(self, data_df, seq_len, pred_len):
        self.data = torch.tensor(data_df.values, dtype=torch.float32)
        self.seq_len = seq_len
        self.pred_len = pred_len
        
    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1
        
    def __getitem__(self, idx):
        X = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return X, y

class TimeSeriesDataset:
    def __init__(self, name, seq_len=24, pred_len=1, train_ratio=0.7, val_ratio=0.2, 
                 batch_size=64, num_workers=0, worker_init_fn=None, generator=None):
        """
        :param num_workers: Số luồng tải dữ liệu
        :param worker_init_fn: Hàm khởi tạo seed cho từng worker
        :param generator: Bộ sinh số ngẫu nhiên của PyTorch
        """
        if name not in DATASET_CONFIG:
            raise ValueError(f"Tên dataset '{name}' không tồn tại trong DATASET_CONFIG.")
        
        self.name = name
        self.config = DATASET_CONFIG[name].copy() 
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.batch_size = batch_size
        
        # Thêm các biến để cấu hình DataLoader tái lập
        self.num_workers = num_workers
        self.worker_init_fn = worker_init_fn
        self.generator = generator
        
        self.df_train = None
        self.df_val = None
        self.df_test = None
        
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        
        self._load_and_split_data()
        
    def _load_and_split_data(self):
        df = pd.read_csv(self.config['path'])
        time_col = self.config['time_col']
        
        if isinstance(time_col, list):
            df['datetime_merged'] = pd.to_datetime(df[time_col])
            df.drop(columns=time_col, inplace=True)
            time_col = 'datetime_merged'
        
        if time_col and time_col in df.columns:
            df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
            df = df.sort_values(by=time_col).reset_index(drop=True)
            df.set_index(time_col, inplace=True)

        if 'No' in df.columns:
            df.drop(columns=['No'], inplace=True)
            
        df = df.select_dtypes(include=[np.number]) # Chỉ giữ lại dữ liệu dạng số
            
        if self.config['type'] == 'univariate':
            df = df[[self.config['target_col']]]
        elif self.config['type'] == 'multivariate':
            self.config['target_col'] = list(df.columns)
            
        df = df.dropna()
            
        n = len(df)
        train_end = int(n * self.train_ratio)
        val_end = int(n * (self.train_ratio + self.val_ratio))
        
        self.df_train = df.iloc[:train_end]
        self.df_val = df.iloc[train_end:val_end]
        self.df_test = df.iloc[val_end:]
        
        self.train_dataset = TSWindowDataset(self.df_train, self.seq_len, self.pred_len)
        self.val_dataset = TSWindowDataset(self.df_val, self.seq_len, self.pred_len)
        self.test_dataset = TSWindowDataset(self.df_test, self.seq_len, self.pred_len)
        
    def get_loaders(self):
        # Đưa worker_init_fn và generator vào DataLoader
        train_loader = DataLoader(
            self.train_dataset, 
            batch_size=self.batch_size, 
            shuffle=True,
            num_workers=self.num_workers,
            worker_init_fn=self.worker_init_fn,
            generator=self.generator
        )
        
        val_loader = DataLoader(
            self.val_dataset, 
            batch_size=self.batch_size, 
            shuffle=False,
            num_workers=self.num_workers,
            worker_init_fn=self.worker_init_fn,
            generator=self.generator
        )
        test_loader = DataLoader(
            self.test_dataset, 
            batch_size=self.batch_size, 
            shuffle=False,
            num_workers=self.num_workers,
            worker_init_fn=self.worker_init_fn,
            generator=self.generator
        )
        return train_loader, val_loader, test_loader
        
    def print_info(self):
        X_tr, y_tr = self.train_dataset[0]
        
        train_batches = len(self.train_dataset) // self.batch_size + (1 if len(self.train_dataset) % self.batch_size != 0 else 0)
        val_batches = len(self.val_dataset) // self.batch_size + (1 if len(self.val_dataset) % self.batch_size != 0 else 0)
        test_batches = len(self.test_dataset) // self.batch_size + (1 if len(self.test_dataset) % self.batch_size != 0 else 0)
        
        print(f"=== THÔNG TIN DATASET: {self.name.upper()} ===")
        print(f"- Phân loại: {self.config['type'].upper()}")
        
        if self.config['type'] == 'multivariate':
            print(f"- Số lượng đặc trưng (Features): {len(self.config['target_col'])} cột")
        else:
            print(f"- Biến mục tiêu (Target): {self.config['target_col']}")
            
        print(f"- Kích thước chuỗi (seq_len, pred_len): ({self.seq_len}, {self.pred_len})")
        print(f"- Batch size: {self.batch_size} | Num workers: {self.num_workers}")
        print("-" * 50)
        print("KÍCH THƯỚC DỮ LIỆU HUẤN LUYỆN (TENSORS & DATALOADER):")
        
        print(f" 1. Tập Train ({self.train_ratio*100:.0f}%):")
        print(f"    + Số cửa sổ: {len(self.train_dataset)} -> Chia thành {train_batches} batches")
        print(f"    + Shape X 1 mẫu : ({X_tr.shape[0]}, {X_tr.shape[1]})")

        print(f" 2. Tập Val ({self.val_ratio*100:.0f}%):")
        print(f"    + Số cửa sổ: {len(self.val_dataset)} -> Chia thành {val_batches} batches")
        
        print(f" 3. Tập Test ({(1 - self.train_ratio - self.val_ratio)*100:.0f}%):")
        print(f"    + Số cửa sổ: {len(self.test_dataset)} -> Chia thành {test_batches} batches")
        print("==================================================")
        
            

In [16]:
RANDOM_SEED = 42

SEED_WORKER = set_seed(RANDOM_SEED)

g = torch.Generator()
g.manual_seed(RANDOM_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

Random seed: 42
Device: cuda


**1. Beijing**

In [17]:
def create_dataset(name_dataset ):
    dataset = TimeSeriesDataset(
    name= name_dataset, 
    seq_len=24, 
    pred_len=1, 
    batch_size=32,
    num_workers=2,                
    worker_init_fn=SEED_WORKER,
    generator=g                   
)

    dataset.print_info()

    train_loader, val_loader, test_loader= dataset.get_loaders()
    return train_loader, val_loader, test_loader

In [18]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def wmape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

def prepare_data(loader):
    X_list, y_list = [], []
    for batch_x, batch_y in loader:
        # Làm phẳng batch_x từ (batch, seq_len, features) -> (batch, seq_len * features)
        X_flat = batch_x.view(batch_x.size(0), -1).numpy()
        # batch_y thường là (batch, pred_len, 1), lấy giá trị cuối
        y_flat = batch_y[:, -1, 0].numpy() 
        X_list.append(X_flat)
        y_list.append(y_flat)
    return np.vstack(X_list), np.concatenate(y_list)

# Bước 1: Chuyển đổi dữ liệu từ DataLoader (từ hình image_85cf11.png)
def get_train_test(train_loader, test_loader):
    X_train, y_train = prepare_data(train_loader)
    X_test, y_test = prepare_data(test_loader)
    return X_train, y_train, X_test, y_test

In [19]:

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)



In [20]:
# Bước 4: Hiển thị kết quả
def print_res( X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    wmape_val = wmape(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{'Metric':<10} | {'Value'}")
    print("-" * 25)
    print(f"{'MSE':<10} | {mse:.4f}")
    print(f"{'MAE':<10} | {mae:.4f}")
    print(f"{'RMSE':<10} | {rmse:.4f}")
    print(f"{'WMAPE':<10} | {wmape_val:.4f}")
    print(f"{'R-square':<10} | {r2:.4f}")


In [21]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import numpy as np

def plot_data_distribution(X_train, y_train, X_test, y_test, max_samples=2000):
    """
    Hàm vẽ phân bố dữ liệu an toàn, tự động giới hạn số lượng mẫu để tránh MemoryError.
    """
    # -------------------------------------------------------------------------
    # BƯỚC 1: LẤY MẪU NGẪU NHIÊN ĐỂ TIẾT KIỆM RAM
    # -------------------------------------------------------------------------
    def subsample(X, y, max_pts):
        if X.shape[0] > max_pts:
            idx = np.random.choice(X.shape[0], max_pts, replace=False)
            return X[idx], y[idx]
        return X, y

    X_train_sub, y_train_sub = subsample(X_train, y_train, max_samples)
    X_test_sub, y_test_sub = subsample(X_test, y_test, max_samples)

    # -------------------------------------------------------------------------
    # BƯỚC 2: CẤU HÌNH VÀ VẼ ĐỒ THỊ
    # -------------------------------------------------------------------------
    sns.set_theme(style="whitegrid")
    plt.rcParams['font.size'] = 11
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

    # --- BIỂU ĐỒ 1: Mật độ phân bố của Target ---
    sns.kdeplot(y_train_sub, fill=True, color="#1f77b4", label="y_train (Train)", ax=axes[0], alpha=0.4)
    sns.kdeplot(y_test_sub, fill=True, color="#ff7f0e", label="y_test (Test)", ax=axes[0], alpha=0.4)
    axes[0].set_title("Mật độ phân bố của Biến mục tiêu", fontsize=12, fontweight='bold')
    axes[0].set_xlabel("Giá trị Target")
    axes[0].set_ylabel("Mật độ (Density)")
    axes[0].legend()

    # --- BIỂU ĐỒ 2: Giảm chiều PCA 2D cho Features ---
    # Fit PCA trên lượng dữ liệu đã thu gọn để không bị MemoryError
    pca = PCA(n_components=2, random_state=42)
    X_train_pca = pca.fit_transform(X_train_sub)
    X_test_pca = pca.transform(X_test_sub)

    # Vẽ các điểm dữ liệu (s=8 giúp các chấm nhỏ lại, đỡ rối mắt)
    axes[1].scatter(X_train_pca[:, 0], X_train_pca[:, 1], color='#1f77b4', alpha=0.25, label='X_train', s=8)
    axes[1].scatter(X_test_pca[:, 0], X_test_pca[:, 1], color='#ff7f0e', alpha=0.45, label='X_test', s=8)
    axes[1].set_title("Không gian đặc trưng qua PCA 2D", fontsize=12, fontweight='bold')
    axes[1].set_xlabel("Thành phần chính 1 (PC1)")
    axes[1].set_ylabel("Thành phần chính 2 (PC2)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

# =========================================================================
# GỌI LẠI HÀM: Lúc này bạn chạy lệnh dưới đây sẽ mượt mà ngay lập tức
# plot_data_distribution(X_train, y_train, X_test, y_test)
# =========================================================================

**3. Bitcoin**

In [22]:
train_loader_b, val_loader_b, test_loader_b = create_dataset("bitcoin")
X_train_b, y_train_b, X_test_b, y_test_b=  get_train_test(train_loader_b, test_loader_b)
plot_data_distribution(X_train_b, y_train_b, X_test_b, y_test_b)
print_res( X_train_b, y_train_b, X_test_b, y_test_b)

MemoryError: 

4.**hanoi AIr**

In [ ]:
train_loader_hn, val_loader_hn, test_loader_hn = create_dataset( "hanoi_air_quality")
X_train_hn, y_train_hn, X_test_hn, y_test_hn=  get_train_test(train_loader_hn, test_loader_hn)
plot_data_distribution(X_train_hn, y_train_hn, X_test_hn, y_test_hn)
print_res( X_train_hn, y_train_hn, X_test_hn, y_test_hn)

**5. Sunspots**

In [ ]:
train_loader_s, val_loader_s, test_loader_s = create_dataset("sunspots")
X_train_s, y_train_s, X_test_s, y_test_s =  get_train_test(train_loader_s, test_loader_s)
plot_data_distribution(X_train_s, y_train_s, X_test_s, y_test_s)
print_res( X_train_s, y_train_s, X_test_s, y_test_s)